# Idealness Prover Comparison

This notebook compares four implementations of idealness verification:

| Implementation | Algorithm | Validation | Output |
|----------------|-----------|------------|--------|
| **exact_prover_v3** | Branch-and-bound | Numerical pre-filter + symbolic solve() | Conditions |
| **exact_numerical** | Branch-and-bound | Fixed values (L=1, U=9, P=2) | IDEAL/NOT IDEAL |
| **enumerate_symbolic** | Direct enumeration | Numerical pre-filter + symbolic solve() | Conditions |
| **enumerate_numeric** | Direct enumeration | Numerical test points only | Conditions* |

*enumerate_numeric uses numerical sampling, not algebraic derivation

All analyze the same formulations:
- **SU**: Standard Unary (4 binary variables, equality coupling)
- **RU**: Refined Unary (4 binary variables, inequality coupling)  
- **SB-L**: Simple Binary with Hamming selector (2 binary variables)
- **SB-M**: Simple Binary with Multilinear selector (2 binary + 1 auxiliary)

In [ ]:
import time
import pandas as pd
from IPython.display import display, HTML

## 1. Import All Implementations

In [ ]:
# Import branch-and-bound symbolic prover (v3)
from exact_prover_v3 import (
    create_SU_formulation as create_SU_v3,
    create_RU_formulation as create_RU_v3,
    create_SBL_formulation as create_SBL_v3,
    create_SBM_formulation as create_SBM_v3,
    ExactBBProver as BBProverV3
)
print("✓ exact_prover_v3 (B&B + symbolic solve)")

In [ ]:
# Import branch-and-bound numerical prover
from exact_numerical import (
    create_SU_formulation as create_SU_num,
    create_RU_formulation as create_RU_num,
    create_SBL_formulation as create_SBL_num,
    create_SBM_formulation as create_SBM_num,
    ExactBBProver as BBProverNum
)
print("✓ exact_numerical (B&B + fixed L=1, U=9, P=2)")

In [ ]:
# Import enumeration-based symbolic prover (truly symbolic)
from enumerate_symbolic import (
    create_SU_formulation_symbolic as create_SU_enum_sym,
    create_RU_formulation_symbolic as create_RU_enum_sym,
    create_SBL_formulation_symbolic as create_SBL_enum_sym,
    create_SBM_formulation_symbolic as create_SBM_enum_sym,
    SymbolicIdealnesProver as EnumSymbolicProver
)
print("✓ enumerate_symbolic (enumeration + symbolic solve)")

In [ ]:
# Import enumeration-based numeric prover
from enumerate_numeric import (
    create_SU_formulation_symbolic as create_SU_enum_num,
    create_RU_formulation_symbolic as create_RU_enum_num,
    create_SBL_formulation_symbolic as create_SBL_enum_num,
    create_SBM_formulation_symbolic as create_SBM_enum_num,
    SymbolicIdealnesProver as EnumNumericProver
)
print("✓ enumerate_numeric (enumeration + numerical tests)")

## 2. Run All Provers

In [ ]:
# Storage for results
all_results = {}
all_timings = {}

MODELS = ['SB-L', 'SB-M', 'RU', 'SU']

In [ ]:
# Run B&B Symbolic (v3)
print("Running B&B Symbolic (exact_prover_v3)...")
v3_factories = {'SB-L': create_SBL_v3, 'SB-M': create_SBM_v3, 'RU': create_RU_v3, 'SU': create_SU_v3}
all_results['v3'] = {}
all_timings['v3'] = {}

for name in MODELS:
    start = time.time()
    prover = BBProverV3(v3_factories[name]())
    _, result = prover.prove()
    all_timings['v3'][name] = time.time() - start
    all_results['v3'][name] = {
        'integral': len(result.always_integral_vertices),
        'conditional': len(result.conditional_vertices),
        'fractional': len(result.always_fractional_vertices),
        'is_ideal': result.is_ideal,
        'is_conditional': result.is_conditionally_ideal,
        'conditions': result.ideal_conditions
    }
    print(f"  {name}: done ({all_timings['v3'][name]:.1f}s)")

In [ ]:
# Run B&B Numerical
print("Running B&B Numerical (exact_numerical)...")
num_factories = {'SB-L': create_SBL_num, 'SB-M': create_SBM_num, 'RU': create_RU_num, 'SU': create_SU_num}
all_results['num'] = {}
all_timings['num'] = {}

for name in MODELS:
    start = time.time()
    prover = BBProverNum(num_factories[name]())
    _, result = prover.prove()
    all_timings['num'][name] = time.time() - start
    all_results['num'][name] = {
        'integral': len(result.integer_vertices),
        'fractional': len(result.fractional_vertices),
        'is_ideal': result.is_ideal
    }
    print(f"  {name}: done ({all_timings['num'][name]:.1f}s)")

In [ ]:
# Run Enumeration Symbolic
print("Running Enumeration Symbolic (enumerate_symbolic)...")
enum_sym_factories = {'SB-L': create_SBL_enum_sym, 'SB-M': create_SBM_enum_sym, 'RU': create_RU_enum_sym, 'SU': create_SU_enum_sym}
all_results['enum_sym'] = {}
all_timings['enum_sym'] = {}

for name in MODELS:
    start = time.time()
    prover = EnumSymbolicProver(enum_sym_factories[name]())
    result = prover.prove()
    all_timings['enum_sym'][name] = time.time() - start
    all_results['enum_sym'][name] = {
        'integral': result['always_integral'],
        'conditional': result['conditional'],
        'fractional': result['always_fractional'],
        'is_ideal': result['is_ideal'],
        'is_conditional': result['is_conditionally_ideal'],
        'conditions': result['ideal_conditions']
    }
    print(f"  {name}: done ({all_timings['enum_sym'][name]:.1f}s)")

## 3. Results Comparison

In [ ]:
# Build comparison dataframe
comparison_data = []

for model in MODELS:
    v3 = all_results['v3'][model]
    num = all_results['num'][model]
    es = all_results['enum_sym'][model]
    
    comparison_data.append({
        'Model': model,
        'V3 Integral': v3['integral'],
        'V3 Cond': v3['conditional'],
        'V3 Frac': v3['fractional'],
        'Num Integer': num['integral'],
        'Num Frac': num['fractional'],
        'EnumSym Integral': es['integral'],
        'EnumSym Cond': es['conditional'],
        'EnumSym Frac': es['fractional'],
    })

df = pd.DataFrame(comparison_data)
print("Vertex Counts:")
display(df)

In [ ]:
# Check agreement
print("Agreement Check:")
print("-" * 70)

for model in MODELS:
    v3 = all_results['v3'][model]
    num = all_results['num'][model]
    es = all_results['enum_sym'][model]
    
    # V3 vs Enum Symbolic should match exactly
    v3_es_match = (v3['integral'] == es['integral'] and 
                   v3['conditional'] == es['conditional'] and
                   v3['fractional'] == es['fractional'])
    
    # Numerical should match total vertices (integral + fractional)
    v3_total = v3['integral'] + v3['conditional'] + v3['fractional']
    num_total = num['integral'] + num['fractional']
    
    print(f"{model}: V3 vs EnumSym: {'✓ MATCH' if v3_es_match else '✗ MISMATCH'}")
    print(f"       V3 total={v3_total}, Num total={num_total}")

## 4. Timing Comparison

In [ ]:
timing_data = []
for model in MODELS:
    timing_data.append({
        'Model': model,
        'B&B Symbolic (v3)': f"{all_timings['v3'][model]:.2f}s",
        'B&B Numerical': f"{all_timings['num'][model]:.2f}s",
        'Enum Symbolic': f"{all_timings['enum_sym'][model]:.2f}s",
    })

timing_df = pd.DataFrame(timing_data)
print("Timing:")
display(timing_df)

## 5. Summary

In [ ]:
print("="*70)
print("IDEALNESS CONCLUSIONS")
print("="*70)

for model in MODELS:
    v3 = all_results['v3'][model]
    
    if v3['is_ideal']:
        print(f"\n{model}: ✓ ALWAYS IDEAL")
        print(f"       All {v3['integral']} vertices are integral for all valid (L, U, P)")
    elif v3['is_conditional']:
        print(f"\n{model}: ◐ CONDITIONALLY IDEAL")
        print(f"       {v3['conditions'][0] if v3['conditions'] else 'Condition unknown'}")
    else:
        print(f"\n{model}: ✗ NOT IDEAL")
        print(f"       Has {v3['fractional']} always-fractional vertices")

print("\n" + "="*70)